# Visual Table Assistant — Inference

## Purpose

Recreate a trained model (`best.pt`) and run predictions on specific images so
you can eyeball how the detector behaves on individual cases. This is a
debugging / auditing notebook, not a training or evaluation one.

Two ways to provide images:

1. **Lookup by name** from the prepared dataset. Pass a filename like
   `00012345_cup_plate.jpg`, or just a fragment like `_knife.jpg`, and the
   notebook finds matching images under
   `datasets/table_assistant_yolo/images/`. This mode needs the dataset to be
   restored once per session (DVC pull + extract), so it is optional.
2. **Manual upload**. Drop image files into the Colab file panel (or any path)
   and point the notebook at them. No dataset restore required.

Ground-truth boxes are drawn on top of predictions by default (lookup mode
only, since uploaded images have no labels).

## Prerequisites

1. **A trained model exists on Drive.** `02_training_colab.ipynb` was run at
   least once, so `best.pt` lives under
   `training_outputs/<run_name>/weights/best.pt` on Drive.
2. The Google account mounting Drive in this session has access to that
   `training_outputs/` folder.
3. **Only for lookup-by-name mode**: the Colab Secret
   `GDRIVE_CREDENTIALS_DATA` is configured (same as in the training notebook),
   so DVC can pull the dataset package non-interactively.

## 1. Repository setup and Google Drive mount

Clone (or update) the repo, mount Drive, and define the same canonical paths
the training notebook uses. Absolute paths keep every cell safe to re-run
independently if the Colab session reconnects.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/LucasGVallejos/iaa-visual-table-assistant.git"
REPO_DIR = Path("/content/iaa-visual-table-assistant")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/iaa-table-assistant")
YOLO_OUTPUTS_DIR = DRIVE_PROJECT_DIR / "training_outputs"

In [ ]:
%cd /content

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already present, pulling latest changes...")
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}

from google.colab import drive
drive.mount("/content/drive")

assert YOLO_OUTPUTS_DIR.exists(), (
    f"Missing {YOLO_OUTPUTS_DIR}. Run 02_training_colab.ipynb first so a "
    "trained model is persisted on Drive."
)
print(f"YOLO_OUTPUTS_DIR: {YOLO_OUTPUTS_DIR}")

## 2. Dependency installation

Install the project's pip requirements (`ultralytics`, `opencv`, `matplotlib`,
`pyyaml`, ...).

In [ ]:
!pip install -q -r requirements.txt

## 3. Select and load the model

`MODEL_SIZE` + `RUN_NAME` identify which training run's `best.pt` to load. They
mirror the constants used in `02_training_colab.ipynb`, so to inspect a
different run just change `MODEL_SIZE` (or set `RUN_NAME` explicitly).

In [ ]:
from ultralytics import YOLO

# Pick the run to inspect. Defaults follow the baseline naming convention.
MODEL_SIZE = "m"  # one of: n, s, m, l, x
RUN_NAME = f"baseline_yolo26{MODEL_SIZE}_001"

BEST_PT = YOLO_OUTPUTS_DIR / RUN_NAME / "weights" / "best.pt"
assert BEST_PT.exists(), (
    f"Missing weights: {BEST_PT}. Check RUN_NAME, or run training first. "
    f"Available runs: {[p.name for p in YOLO_OUTPUTS_DIR.iterdir() if p.is_dir()]}"
)

model = YOLO(str(BEST_PT))

print(f"Loaded model: {BEST_PT}")
print(f"Classes ({len(model.names)}): {model.names}")

## 4. (Optional) Restore the dataset for lookup-by-name

Only needed if you want to query images by filename from the prepared
dataset (and overlay their ground-truth boxes). It pulls the DVC package and
extracts it to `datasets/table_assistant_yolo/`, exactly like the training
notebook.

Skip this entire section if you only plan to run inference on **uploaded**
images. The cell is guarded by `USE_DATASET_LOOKUP` so a plain `Run all`
does not download several GB unless you ask for it.

In [ ]:
import os

# Flip to True to enable lookup-by-name against the prepared dataset.
USE_DATASET_LOOKUP = False

DATASET_IMAGES_DIR = REPO_DIR / "datasets" / "table_assistant_yolo" / "images"
DATASET_LABELS_DIR = REPO_DIR / "datasets" / "table_assistant_yolo" / "labels"

if USE_DATASET_LOOKUP:
    from google.colab import userdata

    gdrive_credentials = userdata.get("GDRIVE_CREDENTIALS_DATA")
    if not gdrive_credentials:
        raise RuntimeError(
            "Missing Colab Secret: GDRIVE_CREDENTIALS_DATA. "
            "Required for lookup-by-name; set it or use upload mode instead."
        )
    os.environ["GDRIVE_CREDENTIALS_DATA"] = gdrive_credentials

    !dvc pull datasets/table_assistant_yolo_package.zip.dvc
    !python -m src.data.preparation.restore_dataset_package

    assert DATASET_IMAGES_DIR.exists(), f"Missing {DATASET_IMAGES_DIR} after restore."
    n_images = sum(1 for _ in DATASET_IMAGES_DIR.glob("*.jpg"))
    print(f"Dataset ready for lookup: {n_images} images at {DATASET_IMAGES_DIR}")
else:
    print("USE_DATASET_LOOKUP is False. Lookup-by-name disabled; upload mode only.")

## 5. Inference configuration

Shared knobs for every prediction below:

- `CONF`: confidence threshold. Detections below it are dropped.
- `SHOW_GT`: draw ground-truth boxes (dashed) alongside predictions (solid).
  Only effective for images coming from the dataset, where labels exist.
- `SAVE`: when True, rendered figures are written to
  `outputs/inference/<run_name>/<timestamp>/`.

In [ ]:
from datetime import datetime

CONF = 0.25
SHOW_GT = True
SAVE = False

_run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR = REPO_DIR / "outputs" / "inference" / RUN_NAME / _run_stamp
if SAVE:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"CONF={CONF}  SHOW_GT={SHOW_GT}  SAVE={SAVE}")
if SAVE:
    print(f"Figures will be saved to: {SAVE_DIR}")

## 6. Inference helpers

Three functions:

- `find_image_by_name(query)` resolves a filename or fragment to an image path
  inside the dataset. Match priority: exact name, then prefix, then suffix,
  then substring. If several images match, all candidates are listed and the
  first is used — narrow the query to disambiguate. Thanks to the
  `<seq>_<class_a>_<class_b>.jpg` naming, a query like `_knife.jpg` returns
  images whose only class is knife, which is handy for auditing rare classes.
- `load_ground_truth(image_path)` reads the matching YOLO label file and
  returns boxes in pixel coordinates.
- `predict_and_render(image_path)` runs the model, draws predictions (solid,
  colored per class from `configs/classes.yaml`) and, when available and
  enabled, ground-truth boxes (dashed).

In [ ]:
import cv2
import yaml
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# Per-class RGB colors from configs/classes.yaml (id -> normalized RGB).
with open(REPO_DIR / "configs" / "classes.yaml", "r", encoding="utf-8") as _f:
    _classes = yaml.safe_load(_f)["classes"]
CLASS_COLORS = {int(c["id"]): tuple(v / 255 for v in c["color"]) for c in _classes}


def find_image_by_name(query: str) -> Path:
    """Resolve a filename or fragment to an image path under the dataset."""
    if not DATASET_IMAGES_DIR.exists():
        raise RuntimeError(
            "Dataset not restored. Set USE_DATASET_LOOKUP=True and run section 4, "
            "or use upload mode with an explicit path."
        )
    names = sorted(p.name for p in DATASET_IMAGES_DIR.glob("*.jpg"))
    exact = [n for n in names if n == query]
    prefix = [n for n in names if n.startswith(query)]
    suffix = [n for n in names if n.endswith(query)]
    substr = [n for n in names if query in n]
    matches = exact or prefix or suffix or substr
    if not matches:
        raise FileNotFoundError(f"No dataset image matches '{query}'.")
    if len(matches) > 1:
        print(f"{len(matches)} matches for '{query}' (showing first 10):")
        for n in matches[:10]:
            print(f"  {n}")
        print(f"Using: {matches[0]}")
    return DATASET_IMAGES_DIR / matches[0]


def load_ground_truth(image_path: Path) -> list:
    """Return ground-truth boxes as (class_id, x1, y1, x2, y2) in pixels."""
    label_path = DATASET_LABELS_DIR / f"{image_path.stem}.txt"
    if not label_path.exists():
        return []
    img = cv2.imread(str(image_path))
    h, w = img.shape[:2]
    boxes = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        cid, cx, cy, bw, bh = int(parts[0]), *[float(v) for v in parts[1:]]
        x1 = (cx - bw / 2) * w
        y1 = (cy - bh / 2) * h
        x2 = (cx + bw / 2) * w
        y2 = (cy + bh / 2) * h
        boxes.append((cid, x1, y1, x2, y2))
    return boxes


def predict_and_render(image_path, conf: float = CONF, show_gt: bool = SHOW_GT):
    """Run the model on one image and render predictions (+ optional GT)."""
    image_path = Path(image_path)
    result = model.predict(str(image_path), conf=conf, verbose=False)[0]

    img_rgb = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(img_rgb)
    ax.axis("off")

    # Predictions: solid boxes.
    for box in result.boxes:
        cid = int(box.cls[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        color = CLASS_COLORS.get(cid, (1, 1, 1))
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                edgecolor=color, linewidth=2))
        ax.text(x1, y1 - 4, f"{model.names[cid]} {float(box.conf[0]):.2f}",
                color="white", fontsize=9,
                bbox=dict(facecolor=color, edgecolor="none", pad=1))

    # Ground truth: dashed boxes.
    gt_boxes = load_ground_truth(image_path) if show_gt else []
    for cid, x1, y1, x2, y2 in gt_boxes:
        color = CLASS_COLORS.get(cid, (1, 1, 1))
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                edgecolor=color, linewidth=2, linestyle="--"))

    legend = [
        Line2D([0], [0], color="black", lw=2, label="prediction"),
        Line2D([0], [0], color="black", lw=2, ls="--", label="ground truth"),
    ]
    ax.legend(handles=legend, loc="upper right", fontsize=8)
    ax.set_title(f"{image_path.name}  |  {len(result.boxes)} pred  |  {len(gt_boxes)} gt")

    if SAVE:
        out_path = SAVE_DIR / f"{image_path.stem}_pred.png"
        fig.savefig(out_path, bbox_inches="tight", dpi=120)
        print(f"Saved: {out_path}")
    plt.show()
    return result

## 7. Single-image inference

Set `IMAGE_QUERY` to a dataset filename/fragment (lookup mode), or set
`IMAGE_PATH` to an explicit path of an uploaded image (upload mode). Leave the
one you are not using as `None`.

In [ ]:
# Lookup mode: a dataset filename or fragment (needs USE_DATASET_LOOKUP=True).
IMAGE_QUERY = "_knife.jpg"

# Upload mode: an explicit path to an uploaded image. Takes precedence if set.
IMAGE_PATH = None

if IMAGE_PATH is not None:
    target = Path(IMAGE_PATH)
    assert target.exists(), f"Image not found: {target}"
else:
    target = find_image_by_name(IMAGE_QUERY)

_ = predict_and_render(target)

## 8. Batch inference

Render several images in one pass. Useful for auditing a class systematically:
with the `<seq>_<class>.jpg` naming, a list of suffix queries like
`['_knife.jpg', '_fork.jpg', '_spoon.jpg']` surfaces examples of the rare
tableware classes side by side, so you can judge whether the model is missing
them or confusing them with each other.

Each entry is resolved the same way as single-image mode: a dataset fragment
(lookup) or an explicit path (upload). For lookup fragments that match many
images, only the first match is rendered — widen the list with more specific
fragments to see more.

In [ ]:
BATCH_QUERIES = [
    "_knife.jpg",
    "_fork.jpg",
    "_spoon.jpg",
]

for query in BATCH_QUERIES:
    candidate = Path(query)
    target = candidate if candidate.exists() else find_image_by_name(query)
    _ = predict_and_render(target)